# Example 1 — Full Pipeline: Raw PDV → Spall Strength (CLI)

This notebook walks through a complete end-to-end HELIX Toolbox run:

```
Raw PDV oscilloscope CSV
        ↓  ALPSS
Smoothed velocity trace + uncertainty CSV
        ↓  SPADE
Spall strength / strain rate / shock stress / HEL summary CSV + plots
```

The run is driven entirely by a YAML config file — the same workflow you
would use in batch / HPC mode via the CLI:
```bash
python helix_cli_runner.py --config my_experiment.yml
```

---
**Before running:** update the path variables in the next cell.

In [ ]:
import os, sys

# ── USER PATHS — edit these ────────────────────────────────────────────────
REPO_ROOT    = os.path.abspath("..")          # path to HELIX_Toolbox_v_2/
INPUT_DIR    = "/path/to/your/pdv/csv/files"  # folder of raw oscilloscope CSVs
PARAM_FOLDER = "/path/to/your/param/folder"   # folder with experiment metadata xlsx/csv
OUTPUT_DIR   = os.path.join(os.path.dirname(os.path.abspath("__file__")), "figures", "example_01_output")
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Repo root :", REPO_ROOT)
print("Input dir :", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)

## 1. Build the config programmatically

You can write a config dict in Python and save it as a YAML file,
or just point to your existing `helix_master_config.yml`.

In [ ]:
from helix_analysis_toolbox import save_config_to_file, load_config_from_file

config = {
    "cli_settings": {
        "input_dir":      INPUT_DIR,
        "input_pattern":  "*.csv",
        "output_dir":     OUTPUT_DIR,
        "param_folder":   PARAM_FOLDER,
        "analysis_mode":  "both",       # both | alpss_only | spade_only
        "spade_mode":     "auto",
        "input_files":    None,
        "spade_input_files": None,
        "spade_input_dir":   None,
        "spade_input_pattern": "*--vel-smooth-with-uncert.csv",
    },
    "alpss_config": {
        "save_data":               "yes",
        "display_plots":           "no",
        "save_all_plots":          "no",
        "header_lines":            22,
        "time_to_take":            4e-6,
        "use_robust_iq_detection": True,
        "iq_threshold_factor":     0.8,
        "smoothing_type":          "savgol",
        "smoothing_window_ns":     6.0,
        "savgol_polyorder":        3,
        "use_notch_filter":        False,
        "sample_rate":             1.28e11,
        "save_velocity_smooth_uncert_csv": True,
        "save_results_csv":        True,
        "C0":      3950.0,
        "density": 8960.0,
        "lam":     1.55e-6,
        "theta":   0.0,
    },
    "spade_config": {
        "experiment_velocity_shots": True,
        "experiment_spall_analysis": True,
        "experiment_hel_detection":  True,
        "analysis_model":            "hybrid",
        "spall_detection_method":    "5-segment",
        "spall_start_time_ns":       0.0,
        "spall_end_time_ns":         90.0,
        "threshold_velocity_ms":     5.0,
        "hel_start_time_ns":         0,
        "hel_end_time_ns":           20,
        "minimum_HEL_velocity_expected": 40.0,
        "hel_rdp_epsilon":           1.25,
        "mad_filter_enabled":        True,
        "mad_filter_threshold":      2.0,
        "skip_unknown_material_traces": True,
        "plot_individual":           True,
        "save_summary":              True,
        "show_plots":                False,
    },
    "material_properties": {
        "Cu": {"density": 8960.0, "bulk_wave_speed": 3950.0, "C0": 3950.0, "C_L": 4700.0},
        "Zn": {"density": 7140.0, "bulk_wave_speed": 3700.0, "C0": 3700.0, "C_L": 4200.0},
    },
}

config_path = os.path.join(OUTPUT_DIR, "run_config.yml")
ok, msg = save_config_to_file(config, config_path)
print(msg)

## 2. Run the CLI

We call `helix_cli_runner.py` as a subprocess so the output streams live
to the notebook cell output exactly as it would in a terminal.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable,
     os.path.join(REPO_ROOT, "helix_cli_runner.py"),
     "--config", config_path],
    capture_output=False,   # stream output to notebook cell directly
    text=True,
)
print("\nExit code:", result.returncode)

## 3. Inspect the output summary

In [ ]:
import pandas as pd

spade_dir = os.path.join(OUTPUT_DIR, "SPADE_analysis")
summary_path = os.path.join(spade_dir, "velocity_shots_summary.csv")

if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    print(f"Loaded {len(df)} traces from {summary_path}")
    display(df.head(10))
else:
    print(f"Summary not found at {summary_path} — check run output above for errors.")

## 4. Quick result plots

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if 'df' not in dir() or df is None:
    print("Run the cell above first to load the summary CSV.")
else:
    # Try common column name variants
    def _find(df, *candidates):
        for c in candidates:
            for col in df.columns:
                if c.lower() in col.lower():
                    return col
        return None

    col_stress  = _find(df, 'shock_stress', 'Shock_Stress')
    col_spall   = _find(df, 'spall_strength', 'Spall_Strength')
    col_strrate = _find(df, 'strain_rate', 'Strain_Rate')
    col_mat     = _find(df, 'material', 'Material', 'Sample')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Spall strength vs strain rate
    if col_spall and col_strrate:
        ax = axes[0]
        if col_mat:
            for mat, grp in df.groupby(col_mat):
                ax.scatter(grp[col_strrate], grp[col_spall], label=mat, s=60)
            ax.legend()
        else:
            ax.scatter(df[col_strrate], df[col_spall], s=60)
        ax.set_xlabel("Strain rate (s⁻¹)")
        ax.set_ylabel("Spall strength (GPa)")
        ax.set_title("Spall strength vs strain rate")
        ax.set_xscale("log")

    # Shock stress vs spall strength
    if col_stress and col_spall:
        ax = axes[1]
        if col_mat:
            for mat, grp in df.groupby(col_mat):
                ax.scatter(grp[col_stress], grp[col_spall], label=mat, s=60)
            ax.legend()
        else:
            ax.scatter(df[col_stress], df[col_spall], s=60)
        ax.set_xlabel("Shock stress (GPa)")
        ax.set_ylabel("Spall strength (GPa)")
        ax.set_title("Spall strength vs shock stress")

    plt.tight_layout()
    fig_path = os.path.join("figures", "example_01_summary_plots.png")
    os.makedirs("figures", exist_ok=True)
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {fig_path}")

## 5. Show a generated individual trace plot

HELIX saves a per-trace spall detection plot for every file when
`plot_individual: true`.  Pick one to display here.

In [ ]:
import glob as _glob
from IPython.display import Image, display as ipy_display

# Find the first spall plot
spall_plots = _glob.glob(os.path.join(spade_dir, "spall_plots", "*.png"))
if spall_plots:
    print(f"Found {len(spall_plots)} spall plots — showing the first one:")
    ipy_display(Image(spall_plots[0], width=800))
else:
    print("No spall plots found — check that plot_individual: true in the config.")